In [101]:
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, MatchAny, FieldCondition, Filter, Prefetch, FusionQuery

import pandas as pd
import numpy as np
import openai
import json
import tiktoken
from dotenv import load_dotenv
import os
import voyageai

In [102]:
qdrant_client = QdrantClient(url="http://localhost:6333")
load_dotenv()
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)


In [10]:
dummy_vector = np.zeros(1024).tolist()

payload = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        query=dummy_vector,
        using="voyage-3",
        limit=1000,
        with_payload=["parent_asin"],
        with_vectors=False
)

parent_asin_list = [item.payload['parent_asin'] for item in payload.points]
df_reviews = pd.read_json("/Users/pranjal/Desktop/projects/AI/data/Electronic_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)
df_review_sample = df_reviews[df_reviews['parent_asin'].isin(parent_asin_list)]

In [11]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,As good as advertised,This product is as good as advertised. The cos...,[{'small_image_url': 'https://m.media-amazon.c...,B07TZJ1CMC,B08912RRG5,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,2022-05-22 20:42:27.409,0,True
1,5,"CIYUPE Tablet Pillow Stand, Tablet Holder (grey)","CIYUPE Tablet Pillow Stand, Tablet Holder Dock...",[],B09V28Y3HH,B09V28Y3HH,AHOEIYJJHZ7ITX75BOFQYNXVVJQQ,2022-12-17 14:53:32.718,1,True
2,5,Looks and works Great,Very small and unimposing support. Looks and w...,[{'small_image_url': 'https://m.media-amazon.c...,B0B2WCTGFV,B0B8N1CQD1,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,2023-02-02 16:42:57.140,0,True
3,5,Didn't work with Motorola 5g uw,I read all directions and charged up my earbud...,[],B092M34HQ3,B0C2BVCBRV,AGSMNIT5YGSO35M2YRIE2OZIOSUQ,2022-05-12 01:30:34.627,0,True
4,5,Two Position Holder,"Durable, has two sturdy positions for iPad and...",[],B09WMLKQ8D,B09HKP7N83,AFJHM4DEMY7IU6ZCUBCETLREE43Q,2022-11-10 19:08:51.479,0,True


# Now we will preprocess the data
We will now compute the title + text
but we have to make sure that this title + text length is less then the max tokens our embedding model can embed
Right now voyage-3 can embed 32k tokens with total of 128 text array in one api calls

In [94]:
# Define functions to preprocess reviews data

def preprocess_reviews_data(row):
    return f"{row['title']} {row['text']}"

# this is exact token count as per voyage-3, since tiktoken oonly do for openai models
# we use the cl100k_base, which is good aprox for voyage-3 
# what we want to achieve here is we want aprox token count per title+text and then use only title+text where len(title+text) < max text length the model can embed
# in our case its voyage-3 which has 32k max length of text length, therefore this does not apply to us
# here we have 2 methods for count of tokens in the title+text
def token_count(row):
        encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(row["preprocessed_data"]))

def approx_token_count(row):
        return int(len(row["preprocessed_data"].split()) * 1.2)

df_review_sample["preprocessed_data"] = df_review_sample.apply(preprocess_reviews_data, axis=1)
df_review_sample["preprocessed_data_token_count"] = df_review_sample.apply(token_count, axis=1)
df_review_sample["preprocessed_data_approx_token_count"] = df_review_sample.apply(approx_token_count, axis=1)
# df_review_sample = df_review_sample[df_review_sample["preprocessed_data_token_count"] < 32000]
df_review_sample = df_review_sample[df_review_sample["preprocessed_data_approx_token_count"] < 32000]


In [95]:
df_review_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,preprocessed_data_token_count,preprocessed_data_approx_token_count
0,5,As good as advertised,This product is as good as advertised. The cos...,[{'small_image_url': 'https://m.media-amazon.c...,B07TZJ1CMC,B08912RRG5,AHITBJSS7KYUBVZPX7M2WJCOIVKQ,2022-05-22 20:42:27.409,0,True,As good as advertised This product is as good ...,43,45
1,5,"CIYUPE Tablet Pillow Stand, Tablet Holder (grey)","CIYUPE Tablet Pillow Stand, Tablet Holder Dock...",[],B09V28Y3HH,B09V28Y3HH,AHOEIYJJHZ7ITX75BOFQYNXVVJQQ,2022-12-17 14:53:32.718,1,True,"CIYUPE Tablet Pillow Stand, Tablet Holder (gre...",104,78
2,5,Looks and works Great,Very small and unimposing support. Looks and w...,[{'small_image_url': 'https://m.media-amazon.c...,B0B2WCTGFV,B0B8N1CQD1,AEVPPTMG43C6GWSR7I2UGRQN7WFQ,2023-02-02 16:42:57.140,0,True,Looks and works Great Very small and unimposin...,17,15
3,5,Didn't work with Motorola 5g uw,I read all directions and charged up my earbud...,[],B092M34HQ3,B0C2BVCBRV,AGSMNIT5YGSO35M2YRIE2OZIOSUQ,2022-05-12 01:30:34.627,0,True,Didn't work with Motorola 5g uw I read all dir...,40,36
4,5,Two Position Holder,"Durable, has two sturdy positions for iPad and...",[],B09WMLKQ8D,B09HKP7N83,AFJHM4DEMY7IU6ZCUBCETLREE43Q,2022-11-10 19:08:51.479,0,True,"Two Position Holder Durable, has two sturdy po...",18,18


In [97]:
total_tokens = df_review_sample["preprocessed_data_approx_token_count"].sum()
total_tokens

np.int64(5274020)

### Create new Qdrant for reviews

In [98]:
qdrant_client.create_collection(
        collection_name="Amazon-items-collection-01-reviews",
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
) 

True

In [99]:
qdrant_client.create_payload_index(
        collection_name="Amazon-items-collection-01-reviews",
        field_name="parent_asin",                 # we will be running many queries
        field_schema=PayloadSchemaType.KEYWORD,   # This will be an exact match
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [103]:
def get_embedding(text, model='voyage-3'):
        result = voyageai_client.embed(
                [text],
                model=model,
        )
        return result.embeddings[0]

def get_embedding_batch(text_list, model='voyage-3', batch_size=100):
        embeddings = []
        counter=1
        for i in range(0, len(text_list), batch_size):
                batch = text_list[i:i + batch_size]
                result = voyageai_client.embed(batch, model=model)
                embeddings.extend(result.embeddings)
                print(f"Processed {counter * batch_size} of {len(text_list)}")
                counter += 1
        return embeddings

### Embed the title+text

In [104]:
data_to_embed_reviews = df_review_sample[['preprocessed_data', 'parent_asin']].to_dict(orient='records')
text_to_embed_reviews = [data['preprocessed_data'] for data in data_to_embed_reviews]
embeddings_reviews = get_embedding_batch(text_to_embed_reviews)

Processed 100 of 91450
Processed 200 of 91450
Processed 300 of 91450
Processed 400 of 91450
Processed 500 of 91450
Processed 600 of 91450
Processed 700 of 91450
Processed 800 of 91450
Processed 900 of 91450
Processed 1000 of 91450
Processed 1100 of 91450
Processed 1200 of 91450
Processed 1300 of 91450
Processed 1400 of 91450
Processed 1500 of 91450
Processed 1600 of 91450
Processed 1700 of 91450
Processed 1800 of 91450
Processed 1900 of 91450
Processed 2000 of 91450
Processed 2100 of 91450
Processed 2200 of 91450
Processed 2300 of 91450
Processed 2400 of 91450
Processed 2500 of 91450
Processed 2600 of 91450
Processed 2700 of 91450
Processed 2800 of 91450
Processed 2900 of 91450
Processed 3000 of 91450
Processed 3100 of 91450
Processed 3200 of 91450
Processed 3300 of 91450
Processed 3400 of 91450
Processed 3500 of 91450
Processed 3600 of 91450
Processed 3700 of 91450
Processed 3800 of 91450
Processed 3900 of 91450
Processed 4000 of 91450
Processed 4100 of 91450
Processed 4200 of 91450
P

In [105]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings_reviews, data_to_embed_reviews):
        pointstructs.append(
                PointStruct(
                        id=i,
                        vector=embedding,
                        payload={
                                'text': data['preprocessed_data'],
                                'parent_asin': data['parent_asin']
                        }
                )
        )
        i += 1

In [106]:
# put the embedding in qdrant
counter = 1
for i in range(0, len(pointstructs), 100):
        batch = pointstructs[i: i+100]
        qdrant_client.upsert(
                collection_name='Amazon-items-collection-01-reviews',
                wait=True,
                points=batch,
        )
        print(f"Processed {counter*1000} of {len(pointstructs)}")

Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 of 91450
Processed 1000 o

In [107]:
# function to run search for reviews on set of parent_asin
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    return results

In [112]:
res = retrieve_prefiltered_reviews_data('bad quality reviews',['B08912RRG5'])

In [113]:
res.points

[ScoredPoint(id=3141, version=34, score=0.5, payload={'text': 'Bad product Poor quality', 'parent_asin': 'B08912RRG5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=672, version=9, score=0.33333334, payload={'text': 'good product good quality', 'parent_asin': 'B08912RRG5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3567, version=38, score=0.25, payload={'text': 'Good quality Fit perfect', 'parent_asin': 'B08912RRG5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4701, version=50, score=0.2, payload={'text': 'Good quality Good quality product', 'parent_asin': 'B08912RRG5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4771, version=50, score=0.16666667, payload={'text': 'The quality and the price Amazing', 'parent_asin': 'B08912RRG5'}, vector=None, shard_key=None, order_value=None)]